In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import vstack

In [ ]:
class LongTextVectorSearch:
    """
    Performs vector-based similarity search on long text documents.

    Workflow:
    1. Split long documents into smaller chunks.
    2. Convert each chunk into a TF-IDF vector.
    3. Store all vectors in a vector database.
    4. Convert the query into a vector.
    5. Compute cosine similarity between the query and all chunks.
    6. Return the most similar chunks.
    """

    def __init__(self, chunk_size=100):
        """
        Initialize the vector search system.
        """

        # Create the TF-IDF vectorizer
        self.vectorizer = TfidfVectorizer()

        # Stores the vectors of all text chunks
        self.vector_db = []

        # Number of words in each chunk
        self.chunk_size = chunk_size

        # Stores the chunk identifiers
        self.chunk_texts = []

    def split_text(self, text):
        """
        Split a long document into smaller chunks.
        """

        # Split the document into words
        words = text.split()

        # Create chunks of 'chunk_size' words
        return [
            ' '.join(words[i:i + self.chunk_size])
            for i in range(0, len(words), self.chunk_size)
        ]

    def create_vector_db(self, texts):
        """
        Create a vector database from multiple text documents.
        """

        # Stores all text chunks
        all_chunks = []

        # Process each document
        for i, text in enumerate(texts):

            # Split the document into chunks
            chunks = self.split_text(text)

            # Add the chunks to the master list
            all_chunks.extend(chunks)

            # Create a unique ID for every chunk
            self.chunk_texts.extend([
                f"text_{i}_chunk_{j}"
                for j in range(len(chunks))
            ])

            print(f"Added {len(chunks)} chunks for text_{i}")

        # Learn the vocabulary from all chunks
        self.vectorizer.fit(all_chunks)

        # Convert every chunk into a TF-IDF vector
        self.vector_db = self.vectorizer.transform(all_chunks)

        # Display some statistics
        print(f"Total chunks: {len(self.chunk_texts)}")
        print(f"Vector shape: {self.vector_db.shape}")
        print(f"Vocabulary size: {len(self.vectorizer.get_feature_names_out())}")

    def search_similar_texts(self, query_text, top_k=5):
        """
        Search for the most similar text chunks.
        """

        # Convert the query into a TF-IDF vector
        query_vector = self.vectorizer.transform([query_text])

        print(f"Generated query vector for: '{query_text[:50]}...'")

        # Calculate cosine similarity between the query
        # and every chunk in the vector database
        similarities = cosine_similarity(
            query_vector,
            self.vector_db
        ).flatten()

        print(similarities[:5])

        # Get the indices of the top-k most similar chunks
        top_indices = similarities.argsort()[-top_k:][::-1]

        # Store the search results
        results = []

        # Retrieve the chunk IDs and similarity scores
        for index in top_indices:

            # Chunk identifier
            chunk_id = self.chunk_texts[index]

            # Similarity score
            similarity = similarities[index]

            # Store the result
            results.append((chunk_id, similarity))

            print(
                f"Similarity between query and "
                f"{chunk_id}: {similarity:.4f}"
            )

        # Return the top matching chunks
        return results

In [ ]:

# Example with longer texts
texts = [
"""What Is a Vector Database?

A vector database is a specialized data storage and retrieval system designed to handle high-dimensional vector data efficiently. In the context of machine learning and artificial intelligence, vectors are numerical representations of data objects, such as words, images, or user behaviors, encoded in a multi-dimensional space. These vectors, often called embeddings, capture the semantic or contextual meaning of the data.

Key Characteristics:

Storage of High-Dimensional Data: Capable of storing vectors with hundreds or thousands of dimensions.
Efficient Similarity Search: Optimized for operations like nearest neighbor search, which finds vectors similar to a given query vector.
Scalability: Designed to handle large volumes of data, potentially billions of vectors.
Integration with AI Models: Works seamlessly with machine learning models that generate embeddings.
Differences Between Vector Databases and Traditional Databases

Data Representation:

Traditional Databases: Store structured data (tables with rows and columns) or unstructured data (documents).
Vector Databases: Store data as high-dimensional vectors representing complex data relationships.
Query Mechanisms:

Traditional Databases: Use SQL queries or key-value lookups for exact matches or range queries.
Vector Databases: Perform similarity searches using distance metrics (e.g., cosine similarity, Euclidean distance).
Indexing Structures:

Traditional Databases: Use B-trees, hash indexes, or inverted indexes.
Vector Databases: Employ specialized indexes like k-d trees, VP-trees, or approximate nearest neighbor algorithms.
Performance Optimization:

Traditional Databases: Optimize for transaction throughput and ACID properties.
Vector Databases: Optimize for low-latency similarity searches over large datasets.
Importance and Applications of Vector Databases

Handling Complex Data: Enable the storage and retrieval of data types that are difficult to manage with traditional databases (e.g., images, audio, text semantics).
Real-Time Recommendations: Provide immediate suggestions based on user interactions by quickly finding similar items.
Enhanced Search Capabilities: Allow semantic search, where queries return results based on meaning rather than keyword matching.
Machine Learning Integration: Serve as a backend for AI applications that require fast access to vector representations.
Use Cases in Industries

E-commerce:

Product Recommendations: Suggest products similar to those a user has viewed or purchased.
Visual Search: Allow users to search for products using images.
Media and Entertainment:

Content Recommendations: Suggest movies, songs, or articles based on user preferences.
Image and Video Retrieval: Find visually similar media content.
Healthcare:

Medical Imaging: Compare patient scans to identify anomalies or similar cases.
Genomics: Analyze genetic data represented as vectors.
Finance:

Fraud Detection: Identify unusual transactions by comparing transaction vectors.
Risk Assessment: Evaluate investment similarities and portfolio diversification."""  # ~650 words
]

In [ ]:
search_engine = LongTextVectorSearch(chunk_size=50)  # 50 words per chunk

print("Creating vector database...")
search_engine.create_vector_db(texts)

In [ ]:
query_text = "What is a Pokemon?"
print(f"\nPerforming similarity search with query text: '{query_text}'")
results = search_engine.search_similar_texts(query_text)

print("\nTop 5 similar chunks:")
for chunk_id, similarity in results:
    print(f"{chunk_id}: Similarity = {similarity:.4f}")